<a href="https://colab.research.google.com/github/oduntanfolake/FlyRank-ML-first-assignment-solution-/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/oduntanfolake/FlyRank-ML-first-assignment-solution-/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [22]:
from google.colab import userdata
from huggingface_hub import HfApi

token = userdata.get("HF_TOKEN")

print("Token loaded:", token is not None)
print("Token length:", len(token) if token else 0)

api = HfApi(token=token)
me = api.whoami()

print("Hugging Face login:", me["name"])

Token loaded: True
Token length: 37
Hugging Face login: Fbeva


In [3]:
import duckdb

con = duckdb.connect()

con.execute(
    "CREATE SECRET (TYPE huggingface, TOKEN ?)",
    [token]
)

print("DuckDB → Hugging Face connection ready.")

DuckDB → Hugging Face connection ready.


In [4]:
import duckdb

con = duckdb.connect()

con.execute(
    "CREATE OR REPLACE SECRET hf_token (TYPE huggingface, TOKEN ?)",
    [token]
)

print("DuckDB connected to Hugging Face.")

DuckDB connected to Hugging Face.


# **Signal B — CTR vs position**

#Verdict: MIXED

CTR varies strongly across position buckets, but most rows in the upper-position buckets have fewer than 100 impressions. Therefore, the apparent CTR pattern is heavily affected by low-volume observations, so position/CTR alone is not reliable enough to use without a volume floor.

In [5]:
ctr_position_check = con.sql("""
WITH base AS (
    SELECT
        gsc_impressions,
        gsc_clicks,
        gsc_sum_position,
        CASE
            WHEN gsc_impressions > 0
            THEN 100.0 * gsc_clicks / gsc_impressions
            ELSE NULL
        END AS ctr
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
    )
    WHERE gsc_impressions > 0
),
bucketed AS (
    SELECT
        CASE
            WHEN gsc_sum_position <= 3 THEN 'top_3'
            WHEN gsc_sum_position <= 10 THEN 'page_1'
            WHEN gsc_sum_position <= 20 THEN 'striking'
            WHEN gsc_sum_position <= 50 THEN 'page_3_5'
            ELSE 'deep'
        END AS position_bucket,
        gsc_clicks,
        ctr
    FROM base
)
SELECT
    position_bucket,
    COUNT(*) AS n,
    COUNT(*) FILTER (WHERE gsc_clicks > 0) AS rows_with_clicks,
    ROUND(
        100.0 * COUNT(*) FILTER (WHERE gsc_clicks > 0) / COUNT(*),
        2
    ) AS pct_with_clicks,
    ROUND(
        AVG(ctr) FILTER (WHERE gsc_clicks > 0),
        3
    ) AS avg_ctr_when_clicked
FROM bucketed
GROUP BY position_bucket
ORDER BY
    CASE position_bucket
        WHEN 'top_3' THEN 1
        WHEN 'page_1' THEN 2
        WHEN 'striking' THEN 3
        WHEN 'page_3_5' THEN 4
        ELSE 5
    END
""")

ctr_position_check.show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────────┬─────────┬──────────────────┬─────────────────┬──────────────────────┐
│ position_bucket │    n    │ rows_with_clicks │ pct_with_clicks │ avg_ctr_when_clicked │
│     varchar     │  int64  │      int64       │     double      │        double        │
├─────────────────┼─────────┼──────────────────┼─────────────────┼──────────────────────┤
│ top_3           │  239747 │             2633 │             1.1 │               61.332 │
│ page_1          │  265419 │             2851 │            1.07 │               41.856 │
│ striking        │  215128 │             3567 │            1.66 │               23.328 │
│ page_3_5        │  346477 │            10262 │            2.96 │               11.415 │
│ deep            │ 2544290 │           398668 │           15.67 │                1.584 │
└─────────────────┴─────────┴──────────────────┴─────────────────┴──────────────────────┘



In [6]:
volume_floor_check = con.sql("""
WITH base AS (
    SELECT
        gsc_impressions,
        gsc_clicks,
        gsc_sum_position,
        CASE
            WHEN gsc_sum_position <= 3 THEN 'top_3'
            WHEN gsc_sum_position <= 10 THEN 'page_1'
            WHEN gsc_sum_position <= 20 THEN 'striking'
            WHEN gsc_sum_position <= 50 THEN 'page_3_5'
            ELSE 'deep'
        END AS position_bucket,
        CASE
            WHEN gsc_impressions >= 100 THEN '100+ impressions'
            ELSE '<100 impressions'
        END AS volume_bucket
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
    )
    WHERE gsc_impressions > 0
)
SELECT
    position_bucket,
    volume_bucket,
    COUNT(*) AS n,
    COUNT(*) FILTER (WHERE gsc_clicks > 0) AS rows_with_clicks
FROM base
GROUP BY position_bucket, volume_bucket
ORDER BY
    CASE position_bucket
        WHEN 'top_3' THEN 1
        WHEN 'page_1' THEN 2
        WHEN 'striking' THEN 3
        WHEN 'page_3_5' THEN 4
        ELSE 5
    END,
    volume_bucket
""")

volume_floor_check.show()

┌─────────────────┬──────────────────┬─────────┬──────────────────┐
│ position_bucket │  volume_bucket   │    n    │ rows_with_clicks │
│     varchar     │     varchar      │  int64  │      int64       │
├─────────────────┼──────────────────┼─────────┼──────────────────┤
│ top_3           │ 100+ impressions │     145 │               10 │
│ top_3           │ <100 impressions │  239602 │             2623 │
│ page_1          │ 100+ impressions │     186 │               14 │
│ page_1          │ <100 impressions │  265233 │             2837 │
│ striking        │ 100+ impressions │     352 │               41 │
│ striking        │ <100 impressions │  214776 │             3526 │
│ page_3_5        │ 100+ impressions │    1432 │              201 │
│ page_3_5        │ <100 impressions │  345045 │            10061 │
│ deep            │ 100+ impressions │  636493 │           271111 │
│ deep            │ <100 impressions │ 1907797 │           127557 │
├─────────────────┴──────────────────┴─────────┴

# **Signal A — Staleness check**
# Analysis
The number of rows in each staleness bucket is very different, so comparing only the raw number of rows with clicks would be misleading. To make a fair comparison, we calculate the percentage of rows within each bucket that received at least one click:

% with clicks = rows_with_clicks / n × 100

Using the results above:

<90 days: 417,260 / 9,617,010 × 100 ≈ 4.34%
90–179 days: 696 / 134,754 × 100 ≈ 0.52%
180–364 days: 25 / 89,614 × 100 ≈ 0.03%

This shows a strong downward pattern. As the content becomes older, the proportion of rows receiving at least one click decreases substantially. The percentage falls from 4.34% for content updated within 90 days to 0.52% for 90–179 days and 0.03% for 180–364 days.

This supports the idea that content staleness is a useful signal for identifying pages that may deserve engagement investigation or refresh attention.

# **Verdict: CONFIRMED**

The proportion of rows with clicks decreases substantially as content becomes older: 4.34% for content updated within 90 days, 0.52% for 90–179 days, and 0.03% for 180–364 days. This supports staleness as a useful signal for prioritizing engagement opportunities.
---



In [15]:
staleness_check = con.sql("""
WITH base AS (
    SELECT
        f.gsc_impressions,
        f.gsc_clicks,

        DATE_DIFF(
            'day',
            c.content_updated_date,
            f.report_date
        ) AS days_since_update

    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet'
    ) f

    LEFT JOIN read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/dim_content.parquet'
    ) c
    ON f.client_hash_id = c.client_hash_id
    AND f.content_hash_id = c.content_hash_id
)

SELECT
    CASE
        WHEN days_since_update IS NULL THEN 'unknown'
        WHEN days_since_update < 90 THEN '<90 days'
        WHEN days_since_update < 180 THEN '90-179 days'
        WHEN days_since_update < 365 THEN '180-364 days'
        ELSE '365+ days'
    END AS staleness_bucket,

    COUNT(*) AS n,

    COUNT(*) FILTER (
        WHERE gsc_clicks > 0
    ) AS rows_with_clicks

FROM base

GROUP BY staleness_bucket

ORDER BY
    CASE staleness_bucket
        WHEN '<90 days' THEN 1
        WHEN '90-179 days' THEN 2
        WHEN '180-364 days' THEN 3
        WHEN '365+ days' THEN 4
        ELSE 5
    END
""")

staleness_check.show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────────┬─────────┬──────────────────┐
│ staleness_bucket │    n    │ rows_with_clicks │
│     varchar      │  int64  │      int64       │
├──────────────────┼─────────┼──────────────────┤
│ <90 days         │ 9617010 │           417260 │
│ 90-179 days      │  134754 │              696 │
│ 180-364 days     │   89614 │               25 │
└──────────────────┴─────────┴──────────────────┘



## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

I will prioritize content that is both stale and still visible in search. A page is considered stale when it has not been updated for at least 180 days, and visible when it has at least 100 GSC impressions during the March 2026 development window.

The purpose of the rule is to identify pages where a content refresh may be worth investigating: the content is old enough to potentially need attention, while still having enough search visibility for an improvement to matter.

For qualifying pages, the score will use their March 2026 GSC impressions so that pages with greater search visibility are ranked higher.

##Reason code

stale_but_visible — the page is old enough to meet the staleness threshold and still has enough search impressions to be considered visible.

In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
rule_check = con.sql("""
WITH base AS (
    SELECT
        f.report_date,
        f.gsc_impressions,
        c.content_updated_date,

        DATE_DIFF(
            'day',
            c.content_updated_date,
            f.report_date
        ) AS days_since_update

    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet'
    ) f

    LEFT JOIN read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/dim_content.parquet'
    ) c
    ON f.client_hash_id = c.client_hash_id
    AND f.content_hash_id = c.content_hash_id
)

SELECT
    COUNT(*) AS total_rows,

    COUNT(*) FILTER (
        WHERE days_since_update >= 180
    ) AS stale_rows,

    COUNT(*) FILTER (
        WHERE gsc_impressions >= 100
    ) AS visible_rows,

    COUNT(*) FILTER (
        WHERE days_since_update >= 180
          AND gsc_impressions >= 100
    ) AS qualifying_rows

FROM base
""")

rule_check.show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬────────────┬──────────────┬─────────────────┐
│ total_rows │ stale_rows │ visible_rows │ qualifying_rows │
│   int64    │   int64    │    int64     │      int64      │
├────────────┼────────────┼──────────────┼─────────────────┤
│    9841378 │      89614 │       638608 │              38 │
└────────────┴────────────┴──────────────┴─────────────────┘



## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

##Baseline scoring and ranking

I apply the rule defined above to the March 2026 development data. Pages that meet both conditions: stale (days_since_update ≥ 180) and visible (gsc_impressions ≥ 100), receive a score equal to their GSC impressions. Pages that do not meet both conditions receive a score of 0.

The qualifying pages receive the reason code stale_but_visible and the action label review_refresh. The results are ranked by score in descending order and written to work/outputs/baseline_action_score.csv

In [19]:
import os

baseline = con.sql("""
WITH base AS (
    SELECT
        f.client_hash_id,
        f.content_hash_id,
        f.report_date,
        f.gsc_impressions,

        DATE_DIFF(
            'day',
            c.content_updated_date,
            f.report_date
        ) AS days_since_update

    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet'
    ) f

    LEFT JOIN read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/dim_content.parquet'
    ) c
    ON f.client_hash_id = c.client_hash_id
    AND f.content_hash_id = c.content_hash_id
),

page_scores AS (
    SELECT
        client_hash_id,
        content_hash_id,

        MAX(
            CASE
                WHEN days_since_update >= 180
                 AND gsc_impressions >= 100
                THEN gsc_impressions
                ELSE 0
            END
        ) AS score

    FROM base
    GROUP BY
        client_hash_id,
        content_hash_id
)

SELECT
    client_hash_id,
    content_hash_id,
    score,

    CASE
        WHEN score > 0
        THEN 'stale_but_visible'
        ELSE 'not_selected'
    END AS reason_code,

    CASE
        WHEN score > 0
        THEN 'review_refresh'
        ELSE 'no_action'
    END AS action

FROM page_scores
ORDER BY score DESC, client_hash_id, content_hash_id
""").df()

os.makedirs("work/outputs", exist_ok=True)

baseline.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print("Rows in page-level queue:", len(baseline))
print("Qualifying pages:", (baseline["score"] > 0).sum())
print("Unique content pages:", baseline["content_hash_id"].nunique())
print("CSV written to: work/outputs/baseline_action_score.csv")

baseline.head(20)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows in page-level queue: 331437
Qualifying pages: 4
Unique content pages: 331437
CSV written to: work/outputs/baseline_action_score.csv


,client_hash_id,content_hash_id,score,reason_code,action
0,client_c182d11e4862a37d,content_5120dcbbb086843d,1075,stale_but_visible,review_refresh
1,client_c182d11e4862a37d,content_42ce26be1ec6be00,348,stale_but_visible,review_refresh
2,client_c182d11e4862a37d,content_bea86ce3455100b0,247,stale_but_visible,review_refresh
3,client_65de48885f4ef01b,content_fb428c6e1ca78da4,123,stale_but_visible,review_refresh
4,client_0797ff3a1fc9a6a5,content_004e9c4c32e88631,0,not_selected,no_action
5,client_0797ff3a1fc9a6a5,content_0236ef736698e17c,0,not_selected,no_action
6,client_0797ff3a1fc9a6a5,content_025f6cfd3c298870,0,not_selected,no_action
7,client_0797ff3a1fc9a6a5,content_0263d5f9b7a2ecd4,0,not_selected,no_action
8,client_0797ff3a1fc9a6a5,content_02752c6c1c60161f,0,not_selected,no_action
9,client_0797ff3a1fc9a6a5,content_0317b24cc1ff5c5d,0,not_selected,no_action


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*


1. Rank 1: content_5120dcbbb086843d
Action: review_refresh
Reason code: stale_but_visible
Confidence: Moderate — the page is 247 days since update and has 1,075 March impressions, so it clearly meets both rule conditions and receives the highest score.
What would make it wrong: The page may still be performing adequately despite being old, or its impressions may not represent a meaningful engagement opportunity.

2. Rank 2: content_42ce26be1ec6be00
Action: review_refresh
Reason code: stale_but_visible
Confidence: Moderate — the page is 264 days since update and has 348 March impressions, satisfying both thresholds.
What would make it wrong: The page may not need a refresh even though it is old and visible.

3. Rank 3: content_bea86ce3455100b0
Action: review_refresh
Reason code: stale_but_visible
Confidence: Moderate — the page is 232 days since update and has 247 March impressions, so it meets both rule conditions.
What would make it wrong: The observed impressions may not indicate that refreshing the content would improve engagement.

4. Rank 4: content_fb428c6e1ca78da4
Action: review_refresh
Reason code: stale_but_visible
Confidence: Moderate — the page is 231 days since update and has 123 March impressions, which clears both thresholds but only narrowly on visibility.
What would make it wrong: The relatively low impression volume may make a refresh less valuable than the rule suggests.

5. Rank 5: content_004e9c4c32e88631
Action: no_action
Reason code: not_selected
Confidence: High that the rule did not select it because its score is 0 and it does not meet the stale-and-visible conditions.
What would make it wrong: The page could still deserve investigation for reasons that this simple baseline does not capture.

6. Rank 6: content_0236ef736698e17c
Action: no_action
Reason code: not_selected
Confidence: High that it does not qualify under the rule. Its recorded update timing is not stale relative to the March window.
What would make it wrong: Other engagement signals not included in this rule could indicate an opportunity.

7. Rank 7: content_025f6cfd3c298870
Action: no_action
Reason code: not_selected
Confidence: High that it does not qualify because its score is 0.
What would make it wrong: The rule may miss an opportunity that would be visible through other signals.

8. Rank 8: content_0263d5f9b7a2ecd4
Action: no_action
Reason code: not_selected
Confidence: High that it does not qualify because it has only 1 March impression and is not stale.
What would make it wrong: A very small amount of observed traffic may not provide enough evidence either way, so the page could require another type of investigation.

9. Rank 9: content_02752c6c1c60161f
Action: no_action
Reason code: not_selected
Confidence: High that it does not meet the baseline conditions.
What would make it wrong: The page could have an engagement issue that is not represented by our two chosen signals.

10. Rank 10: content_0317b24cc1ff5c5d
Action: no_action
Reason code: not_selected
Confidence: High that it is not selected because its score is 0.
What would make it wrong: The simple rule may overlook pages with weak engagement for reasons other than staleness and visibility.

11. Rank 11: content_044c54ec4adcc4b2
Action: no_action
Reason code: not_selected
Confidence: High that it does not qualify under the rule.
What would make it wrong: A page can be an engagement opportunity without meeting our specific stale-and-visible thresholds.

12. Rank 12: content_04c67f3541177192
Action: no_action
Reason code: not_selected
Confidence: High that it does not qualify because it has 31 impressions and is only 34 days since update.
What would make it wrong: The page could have an engagement problem that the current rule does not capture.

13. Rank 13: content_05acc92c165f4386
Action: no_action
Reason code: not_selected
Confidence: High that it does not qualify because it has 9 impressions and is only 34 days since update.
What would make it wrong: Low visibility does not prove that the page has no engagement opportunity.

14. Rank 14: content_07573a1cc2034981
Action: no_action
Reason code: not_selected
Confidence: High that it does not qualify because its score is 0 and its recorded update date is after the March report date.
What would make it wrong: The unusual date relationship may indicate a data-timing issue that should be investigated separately.

15. Rank 15: content_084680e7da2a2ff9
Action: no_action
Reason code: not_selected
Confidence: High that it does not qualify under the rule.
What would make it wrong: The page could still have an opportunity that our two-signal rule cannot detect.

16. Rank 16: content_0b33d8960857ad90
Action: no_action
Reason code: not_selected
Confidence: High that it does not qualify because it has 0 March impressions and is not stale.
What would make it wrong: The absence of observed impressions in this window does not necessarily mean the page has no problem worth investigating.

17. Rank 17: content_0c3410828632f110
Action: no_action
Reason code: not_selected
Confidence: High that it does not qualify because its score is 0 and the recorded update date is later than the March report date.
What would make it wrong: The future-dated update relationship could reflect a data-timing issue rather than true page status.

18. Rank 18: content_0e62212f207dde7a
Action: no_action
Reason code: not_selected
Confidence: High that it does not qualify under the rule.
What would make it wrong: Other engagement signals could identify an opportunity that this baseline misses.

19. Rank 19: content_0f30e04e709c7b5d
Action: no_action
Reason code: not_selected
Confidence: High that it does not qualify because it has 9 March impressions and is only 34 days since update.
What would make it wrong: The page could have an engagement issue that requires signals beyond this baseline.

20. Rank 20: content_0f8cbfa385fe0085
Action: no_action
Reason code: not_selected
Confidence: High that it does not qualify because its score is 0 and it is not stale relative to the March window.
What would make it wrong: The rule is intentionally narrow and could miss opportunities that are not both stale and visible

In [21]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top20_review = con.sql("""
WITH base AS (
    SELECT
        f.client_hash_id,
        f.content_hash_id,
        f.report_date,
        f.gsc_impressions,
        DATE_DIFF(
            'day',
            c.content_updated_date,
            f.report_date
        ) AS days_since_update
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet'
    ) f
    LEFT JOIN read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/dim_content.parquet'
    ) c
    ON f.client_hash_id = c.client_hash_id
    AND f.content_hash_id = c.content_hash_id
),

page_scores AS (
    SELECT
        client_hash_id,
        content_hash_id,
        MAX(
            CASE
                WHEN days_since_update >= 180
                 AND gsc_impressions >= 100
                THEN gsc_impressions
                ELSE 0
            END
        ) AS score
    FROM base
    GROUP BY client_hash_id, content_hash_id
),

ranked AS (
    SELECT
        *,
        ROW_NUMBER() OVER (
            ORDER BY score DESC, client_hash_id, content_hash_id
        ) AS rank
    FROM page_scores
)

SELECT
    r.rank,
    r.client_hash_id,
    r.content_hash_id,
    r.score,
    MAX(b.days_since_update) AS days_since_update,
    MAX(b.gsc_impressions) AS march_impressions,

    CASE
        WHEN r.score > 0 THEN 'review_refresh'
        ELSE 'no_action'
    END AS action,

    CASE
        WHEN r.score > 0 THEN 'stale_but_visible'
        ELSE 'not_selected'
    END AS reason_code

FROM ranked r
LEFT JOIN base b
    ON r.client_hash_id = b.client_hash_id
    AND r.content_hash_id = b.content_hash_id

WHERE r.rank <= 20

GROUP BY
    r.rank,
    r.client_hash_id,
    r.content_hash_id,
    r.score,
    action,
    reason_code

ORDER BY r.rank
""")

top20_review.show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌───────┬─────────────────────────┬──────────────────────────┬───────┬───────────────────┬───────────────────┬────────────────┬───────────────────┐
│ rank  │     client_hash_id      │     content_hash_id      │ score │ days_since_update │ march_impressions │     action     │    reason_code    │
│ int64 │         varchar         │         varchar          │ int64 │       int64       │       int64       │    varchar     │      varchar      │
├───────┼─────────────────────────┼──────────────────────────┼───────┼───────────────────┼───────────────────┼────────────────┼───────────────────┤
│     1 │ client_c182d11e4862a37d │ content_5120dcbbb086843d │  1075 │               247 │              1075 │ review_refresh │ stale_but_visible │
│     2 │ client_c182d11e4862a37d │ content_42ce26be1ec6be00 │   348 │               264 │               348 │ review_refresh │ stale_but_visible │
│     3 │ client_c182d11e4862a37d │ content_bea86ce3455100b0 │   247 │               232 │               247 │ r

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

###Weak picks:
Rank 4 is the weakest qualifying pick because it has the lowest March 2026 impression volume (123). It still meets the rule's 100-impression visibility threshold, but the lower volume makes the potential impact of a refresh less certain. It should therefore be treated as a review candidate, not an automatic refresh decision.

###Leakage check:
The baseline rule uses only days_since_update and March 2026 gsc_impressions. It does not use trend_direction, trend_pct, future-window performance, or product/data-availability flags as scoring signals. The rule therefore does not use the target or future information to rank pages.

In [23]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Leakage check for the baseline rule

print("Features used by baseline:")
print("- days_since_update")
print("- gsc_impressions")

print("\nExcluded from baseline scoring:")
print("- trend_direction")
print("- trend_pct")
print("- future-month performance")
print("- product/data-availability flags")

print("\nBaseline development window: March 2026")
print("Baseline does not use future-month performance.")

Features used by baseline:
- days_since_update
- gsc_impressions

Excluded from baseline scoring:
- trend_direction
- trend_pct
- future-month performance
- product/data-availability flags

Baseline development window: March 2026
Baseline does not use future-month performance.


In [24]:
# Check that the baseline development window is March 2026
# and identify whether any later dates are present.

window_check = con.sql("""
    SELECT
        MIN(report_date) AS earliest_date,
        MAX(report_date) AS latest_date,
        COUNT(*) FILTER (
            WHERE report_date > DATE '2026-03-31'
        ) AS rows_after_march
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
    )
""")

window_check.show()

┌───────────────┬─────────────┬──────────────────┐
│ earliest_date │ latest_date │ rows_after_march │
│     date      │    date     │      int64       │
├───────────────┼─────────────┼──────────────────┤
│ 2026-03-01    │ 2026-03-31  │                0 │
└───────────────┴─────────────┴──────────────────┘



## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.